# Notebook to Compare Heuristics

In [130]:
import asyncio
import nest_asyncio

import random
import pandas as pd
from time import time

from networkx.algorithms.community import greedy_modularity_communities

In [131]:
random.seed(42)  # For accurate comparison
nest_asyncio.apply()

In [132]:
from src import (
    kempe_greedy,
    welfare_greedy,
)

from src import (
    estimate_influence,
    independent_cascade_community,
)

In [133]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../../data/synthetic/networks')
path_to_results = Path('../../results/barbasi_albert/size_200')

file_name = 'barbasi_albert_200'

In [134]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')

    communities = list(greedy_modularity_communities(graph))

    return graph, communities


graph, communities = asyncio.run(main())

## Comparison of Kempe Greedy and Welfare Greedy

In [135]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
num_sims = 1000

In [136]:
results = []

for alpha in alphas:
    start = time()
    kempe_seeds = kempe_greedy(
        graph=graph,
        k=k,
        probability=p,
        num_simulations=num_sims,
    )
    kempe_time = time() - start

    kempe_influence = estimate_influence(
        graph=graph,
        seeds=kempe_seeds,
        propagation_prob=p,
        num_simulations=num_sims,
    )

    kempe_by_comm = independent_cascade_community(
        graph=graph,
        seeds=kempe_seeds,
        probability=p,
        num_sims=num_sims // 2,
    )
    kempe_by_comm_rounded = {k: round(v, 2) for k, v in kempe_by_comm.items()}

    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_influence(
        graph=graph,
        seeds=welfare_seeds,
        propagation_prob=p,
        num_simulations=num_sims,
    )

    welfare_by_comm = independent_cascade_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_sims=num_sims // 2,
    )
    welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}

    results.append(
        {
            'alpha': alpha,
            'kempe_seeds': kempe_seeds,
            'kempe_time_s': kempe_time,
            'kempe_avg_influence': kempe_influence,
            'kempe_influence_by_community': kempe_by_comm_rounded,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_avg_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'org_total_influence': kempe_influence,
            'fair_total_influence': welfare_influence,
        }
    )

df = pd.DataFrame(results)
df.to_csv(f'{path_to_results}/kempe_welfare_{file_name}_{k}_results.csv', index=False)

Selecting seeds: 100%|██████████| 10/10 [00:24<00:00,  2.46s/it, seeds=10, influenced={0: 0.12, 1: 0.12, 2: 0.16, 3: 0.13, 4: 0.13, 5: 0.12, 6: 0.11, 7: 0.12}]
